# 01-Gaussian Parse and Inspect


**Purpose**: parse a real Gaussian log fixture and inspect final-frame structure, status, and scientific results.

**Input**: `tests/test_files/g16log/2-TS1-Opt.log` from the source repository.

**Expected conclusion**: a non-empty batch, final frame, and full summary confirm that the fixture terminated normally and represents a transition state with one imaginary frequency. This notebook is contributor-oriented supplementary material; end users should start with the Markdown quick start.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("TQDM_DISABLE", "1")

from molop.io import AutoParser


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find repository root containing pyproject.toml")


repo_root = find_repo_root()
fixture_path = repo_root / "tests/test_files/g16log/2-TS1-Opt.log"
relative_fixture_path = fixture_path.relative_to(repo_root)

batch = AutoParser(fixture_path.as_posix(), n_jobs=1)
parsed_file = batch[0]
last_frame = parsed_file[-1]

print("summary_snippet:")
parsed_file.to_summary_df()


A richer summary can be obtained by setting `brief=False`.


In [ ]:
parsed_file.to_summary_df(brief=False)

## File Information


In [ ]:
parsed_file

In [ ]:
parsed_file.model_dump()

Properties with units are represented with [pint](https://pint.readthedocs.io/en/stable/) and support unit conversion.


In [ ]:
parsed_file.running_time

In [ ]:
parsed_file.running_time.to("h")

## Frame Object Information


MolOP integrates [rdkit-dof](https://github.com/gentle1999/rdkit-dof). The default RDKit molecule rendering includes a depth-of-field effect. If you don't like this style, you can disable it via the global configuration.

```python
from molop.config import molopconfig
molopconfig.set_dof_effect_drawer(False)
```


In [ ]:
last_frame.rdmol

### Accessing Data in a Frame


In [ ]:
last_frame.model_dump().keys()

If multiple energy levels are available, `total_energy` automatically selects the highest-level energy.


In [ ]:
last_frame.energies.model_dump()

Thermal properties


In [ ]:
last_frame.thermal_informations.model_dump()

Optimization status


In [ ]:
last_frame.geometry_optimization_status.model_dump()

Or view it as a table


In [ ]:
last_frame.geometry_optimization_status.to_df()

Polarizability-related information


In [ ]:
last_frame.polarizability.model_dump()

`molecular_orbitals` can be indexed like a list to access a specific orbital. MolOP also provides shortcuts for commonly used orbitals, for example:


In [ ]:
last_frame.molecular_orbitals.model_dump().keys()

In [ ]:
last_frame.molecular_orbitals[0].model_dump()

In [ ]:
last_frame.molecular_orbitals.HOMO.model_dump()

In [ ]:
last_frame.molecular_orbitals.LUMO.model_dump()

`vibrations` can also be indexed like a list to access information for a specific vibrational mode, for example:


In [ ]:
last_frame.vibrations.model_dump().keys()

In [ ]:
last_frame.vibrations[0].model_dump()


For other available fields, see [Model Field Map](reference/model_fields.md).


## TS-Specific Features


For transition states (TS), MolOP can use the imaginary frequency information for quick inspection to assess whether a TS is reasonable. Bonds that are likely changing connectivity are marked with dashed lines.


In [ ]:
last_frame.to_diff_rdmol()

MolOP successfully validated a reductive elimination step, where Ni changes oxidation state from +2 to +0.


In [ ]:
from rdkit.Chem import Draw

Draw.MolsToGridImage(last_frame.possible_pre_post_ts(show_3D=True), molsPerRow=2, subImgSize=(500, 500))